# PKU YOLOv8 Detection Baseline in Google Colab

This notebook is for the **PKU COCO detection baseline only**.

- Model: `yolov8n.pt`
- Task: object detection
- Training style: fine-tuning / transfer learning from pretrained weights
- Dataset: `configs/pku_coco_baseline.yaml`
- Scope: PKU only, no DeepPCB, no tiling, no segmentation, no severity yet


## Colab Setup Notes

Use one of these project access options:

1. Put the whole repo in Google Drive and mount Drive in Colab.
2. Clone your GitHub repo into `/content/`.
3. Upload the project folder manually if needed.

The notebook below assumes **Google Drive** by default because it is the safest way to keep datasets, runs, and weights between sessions.


In [1]:
!pip install -q ultralytics==8.4.14 opencv-python pyyaml matplotlib

import platform
import sys
import torch

print('Python:', sys.version)
print('Platform:', platform.platform())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    !nvidia-smi
else:
    print('GPU not detected. In Colab, switch Runtime > Change runtime type > GPU before training.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 67.4 MB/s eta 0:00:00
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
Torch: 2.10.0+cu128
CUDA available: True
Thu Apr  9 15:24:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   

## Mount Google Drive and Open the Repo

Update `PROJECT_ROOT` if your repo folder has a different Drive location.

If you prefer to clone the repo instead of using Drive, skip the mount lines and set `PROJECT_ROOT` to the cloned folder path.


In [2]:
from pathlib import Path

USE_DRIVE = True
PROJECT_ROOT = Path('/content/drive/MyDrive/PCB-Defect-Detector')  # change if needed

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

assert PROJECT_ROOT.exists(), f'Update PROJECT_ROOT to your repo folder: {PROJECT_ROOT}'

%cd $PROJECT_ROOT
print('Working directory:', PROJECT_ROOT)


ValueError: mount failed

## Prepare the PKU YOLO Workspace

This runs the repo's existing PKU preparation logic using the validated COCO config.
It does **not** start training yet.


In [ ]:
!python scripts/pku_yolov8_colab_train.py --prepare-only
!cat configs/pku_yolov8_baseline_data.yaml


## Baseline Training Settings

Keep this first Colab run practical. Start with a moderate baseline run, then adjust later after we confirm the workflow is stable.


In [ ]:
MODEL = 'yolov8n.pt'
EPOCHS = 10
IMGSZ = 640
BATCH = 16
WORKERS = 2
DEVICE = '0'
FRACTION = 1.0
PATIENCE = 20
PROJECT_DIR = 'runs/pku_baseline'
RUN_NAME = 'yolov8n_pku_baseline_colab'

print({
    'model': MODEL,
    'epochs': EPOCHS,
    'imgsz': IMGSZ,
    'batch': BATCH,
    'workers': WORKERS,
    'device': DEVICE,
    'fraction': FRACTION,
    'patience': PATIENCE,
    'project_dir': PROJECT_DIR,
    'run_name': RUN_NAME,
})


In [ ]:
import subprocess

train_cmd = [
    'python', 'scripts/pku_yolov8_colab_train.py',
    '--model', MODEL,
    '--epochs', str(EPOCHS),
    '--imgsz', str(IMGSZ),
    '--batch', str(BATCH),
    '--workers', str(WORKERS),
    '--device', DEVICE,
    '--fraction', str(FRACTION),
    '--patience', str(PATIENCE),
    '--project', PROJECT_DIR,
    '--name', RUN_NAME,
]

print('Running:', ' '.join(train_cmd))
subprocess.run(train_cmd, check=True)


## Check the Training Outputs

Ultralytics may place the run under `runs/detect/...`, so this cell checks both likely output locations.


In [ ]:
from pathlib import Path

candidate_run_dirs = [
    Path(PROJECT_DIR) / RUN_NAME,
    Path('runs/detect') / PROJECT_DIR / RUN_NAME,
]

RUN_DIR = None
for candidate in candidate_run_dirs:
    if candidate.exists():
        RUN_DIR = candidate
        break

assert RUN_DIR is not None, 'Training run folder not found yet.'

print('Run directory:', RUN_DIR)
print('Best weights:', RUN_DIR / 'weights' / 'best.pt')
print('Last weights:', RUN_DIR / 'weights' / 'last.pt')

results_csv = RUN_DIR / 'results.csv'
if results_csv.exists():
    import pandas as pd
    display(pd.read_csv(results_csv).tail())


## Quick Validation / Inference Check

This saves a few predicted validation images to a clean inspection folder and displays several inline.


In [ ]:
from pathlib import Path
from IPython.display import Image, display
from ultralytics import YOLO

best_weights = RUN_DIR / 'weights' / 'best.pt'
assert best_weights.exists(), f'Missing best weights: {best_weights}'

val_dir = Path('data/resources/PCB Defects Detection.v1-pku-market-pcb-ver1.coco/valid')
sample_images = sorted(val_dir.glob('*.jpg'))[:8]
assert sample_images, f'No validation images found in {val_dir}'

pred_root = Path('data/inspection_outputs')
pred_name = 'pku_colab_baseline_predictions'

model = YOLO(str(best_weights))
results = model.predict(
    source=[str(path) for path in sample_images],
    imgsz=IMGSZ,
    conf=0.25,
    device=DEVICE,
    save=True,
    project=str(pred_root),
    name=pred_name,
    exist_ok=True,
    verbose=False,
)

pred_dir = pred_root / pred_name
print('Prediction directory:', pred_dir)
print('Predictions generated for', len(results), 'images')

for image_path in sorted(pred_dir.glob('*.jpg'))[:4]:
    display(Image(filename=str(image_path)))


## Notes

- If the first short Colab run gives weak predictions, that is normal.
- The point of this notebook is to give you a clean PKU-only baseline training workflow in Colab.
- DeepPCB training, dataset merging, tiling, segmentation, severity, and augmentation stay for later steps.
